# Data Preparation Report

**Dataset:** [Jigsaw Unintended Bias in Toxicity Classification](https://www.kaggle.com/competitions/jigsaw-unintended-bias-in-toxicity-classification/data)  
**Scope:** data quality assessment and preprocessing

## Summary

The source dataset was screened for missing required values, blank comments, exact duplicates, invalid labels, repeated text, and identity-annotation coverage. The processed dataset preserves original text and fractional toxicity labels, while adding binary toxicity and exact-text group variables for downstream evaluation.


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)


## Source data

`train.csv` contains comment text, toxicity targets, and identity annotations. The original file remains unchanged; processed files are written separately.


In [3]:
candidates = [Path("data/train.csv"), Path("../data/train.csv")]
data_path = next((path.resolve() for path in candidates if path.is_file()), None)

if data_path is None:
    raise FileNotFoundError(
        "Cannot find train.csv. Put it in the data folder inside your project. "
        "If needed, update the candidates list to its actual location."
    )

df = pd.read_csv(data_path)
required_columns = {"id", "comment_text", "target"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Expected the original train.csv; missing columns: {missing_columns}")

original_rows = len(df)
print(f"Loaded {original_rows:,} comments and {df.shape[1]} columns.")
display(df.head())


Loaded 1,804,874 comments and 45 columns.


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,bisexual,black,buddhist,christian,female,heterosexual,hindu,homosexual_gay_or_lesbian,intellectual_or_learning_disability,jewish,latino,male,muslim,other_disability,other_gender,other_race_or_ethnicity,other_religion,other_sexual_orientation,physical_disability,psychiatric_or_mental_illness,transgender,white,created_date,publication_id,parent_id,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count
0,59848,0.000000,"This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well ...",0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:41.987077+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4
1,59849,0.000000,"Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyon...",0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:42.870083+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4
2,59852,0.000000,This is such an urgent design problem; kudos to you for taking it on. Very impressive!,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:45.222647+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4
3,59855,0.000000,Is this something I'll be able to install on my site? When will you be releasing it?,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:47.601894+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4
4,59856,0.893617,haha you guys are a bunch of losers.,0.021277,0.0,0.021277,0.87234,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-09-29 10:50:48.488476+00,2,NaN,2006,rejected,0,0,0,1,0,0.0,4,47


## Data characteristics

`target` is the share of annotators who classified a comment as toxic, on a 0–1 scale. Identity fields represent identities mentioned in the text, not author characteristics. Missing identity annotations are retained as missing.


In [4]:
df.info()

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (100 * df.isna().mean()).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_summary)


<class 'pandas.DataFrame'>
RangeIndex: 1804874 entries, 0 to 1804873
Data columns (total 45 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   id                                   int64  
 1   target                               float64
 2   comment_text                         str    
 3   severe_toxicity                      float64
 4   obscene                              float64
 5   identity_attack                      float64
 6   insult                               float64
 7   threat                               float64
 8   asian                                float64
 9   atheist                              float64
 10  bisexual                             float64
 11  black                                float64
 12  buddhist                             float64
 13  christian                            float64
 14  female                               float64
 15  heterosexual                         float6

,missing_count,missing_percent
muslim,1399744,77.55
hindu,1399744,77.55
white,1399744,77.55
transgender,1399744,77.55
psychiatric_or_mental_illness,1399744,77.55
physical_disability,1399744,77.55
other_sexual_orientation,1399744,77.55
other_religion,1399744,77.55
other_race_or_ethnicity,1399744,77.55
other_gender,1399744,77.55


In [5]:
identity_columns = [
    "male", "female", "transgender", "other_gender",
    "heterosexual", "homosexual_gay_or_lesbian", "bisexual",
    "other_sexual_orientation", "christian", "jewish", "muslim",
    "hindu", "buddhist", "atheist", "other_religion",
    "black", "white", "asian", "latino", "other_race_or_ethnicity",
    "physical_disability", "intellectual_or_learning_disability",
    "psychiatric_or_mental_illness", "other_disability"
]
identity_columns = [column for column in identity_columns if column in df.columns]
print(f"Found {len(identity_columns)} identity columns.")

display(df["target"].describe())

inspection = pd.Series({
    "missing_comment_text": df["comment_text"].isna().sum(),
    "blank_or_missing_comment_text": df["comment_text"].fillna("").str.strip().eq("").sum(),
    "missing_target": df["target"].isna().sum(),
    "missing_id": df["id"].isna().sum(),
    "duplicate_ids_after_first": df["id"].duplicated().sum(),
    "repeated_comment_text_after_first": df["comment_text"].duplicated().sum()
})
display(inspection.to_frame("count"))


Found 24 identity columns.


count    1.804874e+06
mean     1.030173e-01
std      1.970757e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.666667e-01
max      1.000000e+00
Name: target, dtype: float64

,count
missing_comment_text,3
blank_or_missing_comment_text,3
missing_target,0
missing_id,0
duplicate_ids_after_first,0
repeated_comment_text_after_first,24051


## Cleaning decisions

Records without an ID, comment text, or toxicity target are excluded. Leading and trailing whitespace is removed, then exact duplicate rows are removed. Capitalization, punctuation, emojis, stopwords, identity terms, and fractional labels are retained.

Identical comments with different annotations remain in the data and receive a common `text_group` value.


In [6]:
clean_df = df.dropna(subset=["id", "comment_text", "target"]).copy()
removed_missing = original_rows - len(clean_df)

clean_df["comment_text"] = clean_df["comment_text"].str.strip()
blank_mask = clean_df["comment_text"].eq("")
removed_blank = int(blank_mask.sum())
clean_df = clean_df.loc[~blank_mask].copy()

before_dedup = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
removed_exact_duplicates = before_dedup - len(clean_df)

# Duplicate IDs with different data need investigation, not an arbitrary keep-first rule.
if clean_df["id"].duplicated().any():
    raise ValueError(
        "Some IDs have conflicting records after exact duplicates were removed. "
        "Inspect clean_df[clean_df['id'].duplicated(keep=False)] before continuing."
    )

print(f"Removed {removed_missing:,} rows missing text, target, or ID.")
print(f"Removed {removed_blank:,} blank comments.")
print(f"Removed {removed_exact_duplicates:,} exact duplicate records.")
print(f"Remaining comments: {len(clean_df):,}")


Removed 3 rows missing text, target, or ID.
Removed 0 blank comments.
Removed 0 exact duplicate records.
Remaining comments: 1,804,871


## Target definition

Target and identity variables are validated as numeric values from 0 to 1. For binary summaries, `target >= 0.5` is classified as toxic. The original fractional `target` remains available for modelling.


In [7]:
label_columns = ["target"] + identity_columns
for column in label_columns:
    original = clean_df[column]
    numeric = pd.to_numeric(original, errors="coerce")
    invalid = original.notna() & (numeric.isna() | ~numeric.between(0, 1))
    if invalid.any():
        raise ValueError(f"{column} contains {int(invalid.sum())} invalid labels.")
    clean_df[column] = numeric

if clean_df.empty:
    raise ValueError("No usable comments remain after cleaning.")

clean_df["toxic"] = (clean_df["target"] >= 0.5).astype("int8")

class_summary = (
    clean_df["toxic"].value_counts().reindex([0, 1], fill_value=0)
    .rename_axis("toxic").to_frame("comments")
)
class_summary["percent"] = (100 * class_summary["comments"] / len(clean_df)).round(2)
display(class_summary)
print("0 = non-toxic; 1 = toxic")


,comments,percent
toxic,,
0,1660537,92.0
1,144334,8.0


0 = non-toxic; 1 = toxic


## Descriptive results

An identity score of at least 0.5 is counted as a mention. Missing values are excluded from annotation counts. Because a comment may mention multiple identities, group counts overlap. These tables describe the dataset; they are not fairness-performance results.


In [8]:
identity_rows = []
for column in identity_columns:
    mentioned = clean_df[column].ge(0.5)
    identity_rows.append({
        "identity": column,
        "annotated_comments": int(clean_df[column].notna().sum()),
        "comments_mentioning_identity": int(mentioned.sum()),
        "non_toxic_mentions": int((mentioned & clean_df["toxic"].eq(0)).sum()),
        "toxic_mentions": int((mentioned & clean_df["toxic"].eq(1)).sum())
    })

identity_summary = pd.DataFrame(identity_rows, columns=[
    "identity", "annotated_comments", "comments_mentioning_identity",
    "non_toxic_mentions", "toxic_mentions"
]).sort_values("comments_mentioning_identity", ascending=False)
display(identity_summary)

# Identical text must stay in one split to avoid leaking comments between sets.
# These group IDs apply to this saved dataset; preserve them when creating splits.
clean_df["text_group"] = pd.factorize(clean_df["comment_text"], sort=False)[0]
print(f"Unique cleaned comment texts: {clean_df['text_group'].nunique():,}")
print(f"Additional rows sharing text: {clean_df['text_group'].duplicated().sum():,}")


,identity,annotated_comments,comments_mentioning_identity,non_toxic_mentions,toxic_mentions
1,female,405128,53429,46118,7311
0,male,405128,44484,37799,6685
8,christian,405128,40423,36750,3673
16,white,405128,25082,18044,7038
10,muslim,405128,21006,16225,4781
15,black,405128,14901,10223,4678
5,homosexual_gay_or_lesbian,405128,10997,7876,3121
9,jewish,405128,7651,6411,1240
22,psychiatric_or_mental_illness,405128,4889,3859,1030
17,asian,405128,4578,4012,566


Unique cleaned comment texts: 1,780,822
Additional rows sharing text: 24,049


## Saved artifacts

| File | Contents |
| --- | --- |
| `cleaned_train.csv.gz` | cleaned records, binary target, and text-group ID |
| `cleaning_summary.csv` | record counts by cleaning stage |
| `class_summary.csv` | binary class distribution |
| `identity_summary.csv` | identity-annotation and toxicity counts |


In [9]:
output_dir = data_path.parent / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = output_dir / "cleaned_train.csv.gz"
clean_df.to_csv(cleaned_path, index=False, compression="gzip")
identity_summary.to_csv(output_dir / "identity_summary.csv", index=False)
class_summary.to_csv(output_dir / "class_summary.csv")

cleaning_summary = pd.DataFrame({
    "step": ["original_rows", "removed_missing_required", "removed_blank_text",
             "removed_exact_duplicates", "remaining_rows"],
    "rows": [original_rows, removed_missing, removed_blank,
             removed_exact_duplicates, len(clean_df)]
})
cleaning_summary.to_csv(output_dir / "cleaning_summary.csv", index=False)
display(cleaning_summary)
print(f"Saved cleaned data to: {cleaned_path}")


,step,rows
0,original_rows,1804874
1,removed_missing_required,3
2,removed_blank_text,0
3,removed_exact_duplicates,0
4,remaining_rows,1804871


Saved cleaned data to: /Users/sylviadu/Documents/DATA 4620/Bert-Toxicity-Identification/data/processed/cleaned_train.csv.gz


## Evaluation implications

Exact-text groups are preserved for partitioning, preventing duplicate comments from crossing training and evaluation data. Fixed partitions support a direct comparison between baseline and revised BERT models.
